# Molmo 2: Pointing on Satellite Imagery (HuggingFace Transformers)

**Goal:** Test whether Molmo 2 can **point to** hydraulic features on satellite imagery with pixel coordinates.

**Context:** Molmo 2 is loaded locally via HuggingFace Transformers + PyTorch MPS (Apple Silicon GPU). The model (~5-6 GB) runs directly on M1 Pro 32GB without a server.

Molmo 2 is the only VLM that returns native `(x, y)` point coordinates, making it the ideal bridge to SAM segmentation.

| Model | Capability | Output |
|-------|-----------|--------|
| Gemma 4 | Describe features | Qualitative text |
| **Molmo 2** | **Point to features** | **`<point x="52.3" y="41.7">river channel</point>`** |
| SAM (Stage 7) | Segment from points | Binary mask |

**Roadmap reference:** Stage 6.4 — Molmo 2 Pointing on Satellite (Local)

## Tasks
1. Load Molmo2-8B via HuggingFace Transformers (MPS backend)
2. Test pointing prompts on satellite RGB
3. Parse point coordinates and overlay on satellite image
4. Convert pixel coordinates to geo-coordinates
5. Evaluate: are points accurate enough for SAM prompts?
6. Save timestamped outputs to `data/output/model-outputs/attempt2/molmo2/`

In [ ]:
import os
import io
import re
import json
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

# Paths
DEM_PATH = Path("../../data/input/1m elevation.tif")
SATELLITE_PATH = Path("../../data/output/cimarron_satellite.png")
HILLSHADE_PATH = Path("../../data/output/cimarron_hillshade.png")
OUTPUT_DIR = Path("../../data/output/model-outputs/attempt2/molmo2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Timestamp for this run
RUN_TS = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

print(f"DEM: {DEM_PATH}")
print(f"Satellite image: {SATELLITE_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Run timestamp: {RUN_TS}")
print(f"PyTorch version: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 1.1 Load Molmo 2 via HuggingFace Transformers

Loads `allenai/Molmo2-8B` directly using HuggingFace Transformers with PyTorch MPS backend.

**First run:** Downloads ~5-6 GB of model weights (cached for future runs).

In [ ]:
from transformers import QuantoConfig

MODEL_ID = "allenai/Molmo2-8B"
MOLMO_AVAILABLE = False

device = "mps" if torch.backends.mps.is_available() else "cpu"

try:
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

    quantization_config = QuantoConfig(weights="int4")
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
    ).to(device)
    model.eval()

    # Quick sanity check (text-only)
    test_inputs = processor.apply_chat_template(
        [{"role": "user", "content": [dict(type="text", text="Say hello in one word.")]}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    )
    test_inputs = {k: v.to(device) for k, v in test_inputs.items()}
    with torch.no_grad():
        test_out = model.generate(**test_inputs, max_new_tokens=10)
    test_response = processor.tokenizer.decode(
        test_out[0, test_inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    print(f"Molmo 2 loaded OK: {test_response}")
    print(f"Model: {MODEL_ID} (Quanto int4)")
    print(f"Device: {device}")
    MOLMO_AVAILABLE = True
except Exception as e:
    print(f"Molmo 2 failed to load.")
    print(f"Error: {e}")
    import traceback; traceback.print_exc()
    print()
    print("Setup instructions:")
    print("  1. pip install transformers==4.57.1 accelerate einops molmo_utils optimum-quanto")
    print("  2. Ensure ~6 GB free disk space for model download")
    print("  3. Requires PyTorch with MPS support (Apple Silicon) or CUDA")
    print()
    print("The rest of this notebook will be skipped.")

## 1.2 Load Images & DEM Metadata

In [3]:
# Load DEM metadata for geo-coordinate conversion
with rasterio.open(DEM_PATH) as src:
    dem_transform = src.transform
    dem_bounds = src.bounds
    dem_crs = src.crs

print(f"CRS: {dem_crs}")
print(f"Bounds: {dem_bounds}")

# Load satellite image
if not SATELLITE_PATH.exists():
    raise FileNotFoundError(
        f"Satellite image not found at {SATELLITE_PATH}.\n"
        "Run the satellite acquisition notebook (Stage 5) first."
    )

img_satellite = Image.open(SATELLITE_PATH)
print(f"Satellite image: {img_satellite.size} ({img_satellite.mode})")

# Load hillshade for dual-input comparison
img_hillshade = None
if HILLSHADE_PATH.exists():
    img_hillshade = Image.open(HILLSHADE_PATH)
    print(f"Hillshade image: {img_hillshade.size} ({img_hillshade.mode})")

CRS: EPSG:26914
Bounds: BoundingBox(left=645114.9999575217, bottom=3982702.00003351, right=650420.9999575217, top=3985790.00003351)
Satellite image: (9010, 5313) (RGB)
Hillshade image: (5306, 3088) (RGB)


## 1.3 Inference & Coordinate Parsing Utilities

In [ ]:
def resize_for_inference(img, max_width=1024):
    """Resize image to reduce memory while preserving aspect ratio."""
    if img.width > max_width:
        ratio = max_width / img.width
        new_size = (max_width, int(img.height * ratio))
        return img.resize(new_size, Image.LANCZOS)
    return img


def vision_query(image, prompt, max_new_tokens=512):
    """Send an image + text prompt to Molmo 2 via HuggingFace Transformers."""
    img = resize_for_inference(image)
    messages = [
        {
            "role": "user",
            "content": [
                dict(type="text", text=prompt),
                dict(type="image", image=img),
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    # Decode only the generated tokens (skip the input prompt)
    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    return processor.tokenizer.decode(generated_ids, skip_special_tokens=True)


# Molmo2 coordinate parsing regexes (from official docs)
COORD_REGEX = re.compile(r'<(?:points|tracks)[^>]* coords="([^"]+)"')
POINTS_REGEX = re.compile(r'([0-9]+) ([0-9]{3,4}) ([0-9]{3,4})')


def parse_molmo_points(text, image_w, image_h):
    """Parse Molmo2 coordinate output into pixel coordinates.

    Molmo2 outputs coords as triplets: index x y
    where x,y are 3-4 digit integers on a 0-1000 scale.

    Also handles legacy <point x="..." y="..."> format (0-100 scale).

    Returns list of (x_px, y_px, label) tuples.
    """
    points = []

    # Molmo2 format: <points coords="idx x y idx x y ...">label</points>
    for coord_match in COORD_REGEX.finditer(text):
        coord_str = coord_match.group(1)
        for pt_match in POINTS_REGEX.finditer(coord_str):
            idx = pt_match.group(1)
            x_raw, y_raw = float(pt_match.group(2)), float(pt_match.group(3))
            x_px = x_raw / 1000.0 * image_w
            y_px = y_raw / 1000.0 * image_h
            if 0 <= x_px <= image_w and 0 <= y_px <= image_h:
                points.append((x_px, y_px, f"point_{idx}"))

    # Legacy format: <point x="..." y="...">label</point> (0-100 scale)
    if not points:
        legacy_pattern = r'<point\s+x="([\d.]+)"\s+y="([\d.]+)"(?:\s+alt="[^"]*")?>([^<]*)</point>'
        for match in re.finditer(legacy_pattern, text):
            x_pct, y_pct, label = float(match.group(1)), float(match.group(2)), match.group(3).strip()
            x_px = x_pct / 100.0 * image_w
            y_px = y_pct / 100.0 * image_h
            points.append((x_px, y_px, label))

    return points


def pixels_to_geo(points, transform):
    """Convert pixel coordinates to geo-coordinates."""
    geo_coords = []
    for x_px, y_px, label in points:
        easting, northing = rasterio.transform.xy(transform, int(y_px), int(x_px))
        geo_coords.append((easting, northing, label))
    return geo_coords


def molmo_point(image, prompt):
    """Run a pointing query and parse the results."""
    resized = resize_for_inference(image)
    raw = vision_query(image, prompt)
    pixel_coords = parse_molmo_points(raw, resized.width, resized.height)
    geo_coords = pixels_to_geo(pixel_coords, dem_transform) if pixel_coords else []
    return {
        "raw_output": raw,
        "pixel_coords": pixel_coords,
        "geo_coords": geo_coords,
        "n_points": len(pixel_coords),
    }


def plot_points_on_image(img, points_px, title, save_path=None, figsize=(14, 8)):
    """Overlay parsed points on an image."""
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(img)
    ax.set_title(title, fontsize=14)

    if points_px:
        xs = [p[0] for p in points_px]
        ys = [p[1] for p in points_px]
        labels = [p[2] for p in points_px]
        ax.scatter(xs, ys, c="red", s=80, marker="x", linewidths=2, zorder=5)
        for x, y, label in zip(xs, ys, labels):
            ax.annotate(label, (x, y), textcoords="offset points",
                        xytext=(5, 5), fontsize=8, color="red",
                        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))
        ax.set_xlabel(f"{len(points_px)} points detected")
    else:
        ax.set_xlabel("No coordinate points parsed from model output")

    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()
    return fig


pointing_results = {}
print("Utilities ready.")

## 2.1 Pointing Tests — Satellite

In [5]:
if not MOLMO_AVAILABLE:
    print("Skipping — Molmo 2 is not available.")
else:
    satellite_pointing_prompts = {
        "channel": "Point to the main river channel in this satellite image.",
        "channel_bends": "Point to where the river channel bends or meanders in this satellite image.",
        "roads": "Point to any roads or road embankments visible in this satellite image.",
        "vegetation_boundary": "Point to the boundaries between forest and open grassland in this satellite image.",
        "point_bars": "Point to any sand bars or point bars along the river in this satellite image.",
        "bridges": "Point to any bridges crossing the river in this satellite image.",
    }

    for key, prompt in satellite_pointing_prompts.items():
        test_name = f"satellite_{key}"
        print(f"\n{'='*80}")
        print(f"TEST: {test_name}")
        print(f"Prompt: {prompt}")
        print(f"{'='*80}")

        result = molmo_point(img_satellite, prompt)
        pointing_results[test_name] = {"prompt": prompt, **result}

        print(f"\nRaw output:\n{result['raw_output']}")
        print(f"\nPoints found: {result['n_points']}")
        if result['pixel_coords']:
            for px, py, label in result['pixel_coords']:
                print(f"  ({px:.0f}, {py:.0f}) — {label}")

        save_path = OUTPUT_DIR / f"{RUN_TS}_satellite_{key}.png"
        plot_points_on_image(img_satellite, result['pixel_coords'],
                             f"Molmo 2 Pointing: {key} (satellite)",
                             save_path=save_path)

Skipping — Molmo 2 is not available.


In [ ]:
# Re-parse raw outputs from previous run (avoids re-running inference)
# Remove this cell after a successful full run.
REPARSE_OUTPUTS = {
    "satellite_channel": {
        "prompt": "Point to the main river channel in this satellite image.",
        "raw_output": '<points coords="1 1 351 666">main river channel</points>',
    },
    "satellite_channel_bends": {
        "prompt": "Point to where the river channel bends or meanders in this satellite image.",
        "raw_output": '<points coords="1 1 011 666 2 096 816 3 266 866 4 356 566 5 360 716 6 466 501 7 566 576 8 606 656 9 666 666 10 736 576 11 816 466 12 876 366 13 956 216">where the river channel bends or meanders in this satellite image</points>',
    },
    "satellite_roads": {
        "prompt": "Point to any roads or road embankments visible in this satellite image.",
        "raw_output": '<points coords="1 1 026 436 2 036 116 3 066 436 4 076 116 5 086 566 6 106 566 7 116 576 8 156 576 9 166 576 10 186 576 11 206 576 12 226 576 13 246 576 14 266 576 15 286 576 16 306 576 17 326 576 18 346 576 19 366 576 20 386 576 21 406 576 22 426 576 23 446 576 24 466 576 25 486 576 26 506 576 27 526 576 28 546 576 29 566 576 30 586 576 31 606 576 32 626 576 33 646 576 34 666 576 35 686 576 36 706 576 37 726 576 38 746 576 39 766 576 40 786 576 41 806 576 42 826 576 43 846 576 44 866 576 45 886 576 46 906 576 47 926 576">roads or road embankments</points>',
    },
    "satellite_vegetation_boundary": {
        "prompt": "Point to the boundaries between forest and open grassland in this satellite image.",
        "raw_output": '<points coords="1 1 066 316 2 166 616 3 256 336 4 466 666 5 556 336 6 726 176 7 916 416">boundaries between forest and open grassland</points>',
    },
    "satellite_point_bars": {
        "prompt": "Point to any sand bars or point bars along the river in this satellite image.",
        "raw_output": '<points coords="1 1 026 716 2 076 766 3 336 796 4 356 616 5 566 606 6 666 666 7 826 456 8 896 336 9 966 216">sand bars or point bars along the river in this satellite image.</points>',
    },
    "satellite_bridges": {
        "prompt": "Point to any bridges crossing the river in this satellite image.",
        "raw_output": '<points coords="1 1 626 666 2 656 676">bridges crossing the river in this satellite image.</points>',
    },
}

resized = resize_for_inference(img_satellite)
pointing_results = {}

for key, data in REPARSE_OUTPUTS.items():
    raw = data["raw_output"]
    pixel_coords = parse_molmo_points(raw, resized.width, resized.height)
    geo_coords = pixels_to_geo(pixel_coords, dem_transform) if pixel_coords else []
    pointing_results[key] = {
        "prompt": data["prompt"],
        "raw_output": raw,
        "pixel_coords": pixel_coords,
        "geo_coords": geo_coords,
        "n_points": len(pixel_coords),
    }

    print(f"\n{'='*60}")
    print(f"{key}: {len(pixel_coords)} points parsed")
    if pixel_coords:
        for px, py, label in pixel_coords:
            print(f"  ({px:.0f}, {py:.0f}) — {label}")

    save_path = OUTPUT_DIR / f"{RUN_TS}_reparse_{key}.png"
    plot_points_on_image(
        resize_for_inference(img_satellite),
        pixel_coords,
        f"Molmo 2 Pointing: {key.replace('satellite_', '')} (satellite)",
        save_path=save_path,
    )

print(f"\nTotal points parsed: {sum(r['n_points'] for r in pointing_results.values())}")

## 3. Results Summary & Save

In [ ]:
if not MOLMO_AVAILABLE:
    print("Molmo 2 was not available — no results to summarize.")
else:
    # Pointing results table
    print("POINTING RESULTS")
    print(f"{'Test':<30} {'Points':>6}  {'Coordinates (pixel)'}")
    print("-" * 80)
    for key, result in pointing_results.items():
        n = result["n_points"]
        if result["pixel_coords"]:
            coords_str = "; ".join(f"({p[0]:.0f},{p[1]:.0f})" for p in result["pixel_coords"][:5])
            if n > 5:
                coords_str += f" ... (+{n - 5} more)"
        else:
            coords_str = "(no coordinates parsed)"
        print(f"{key:<30} {n:>6}  {coords_str}")

    print(f"\nModel: {MODEL_ID} (HuggingFace Transformers, local)")
    print(f"Total pointing tests: {len(pointing_results)}")

    # Save results to markdown
    md_lines = [
        f"# Molmo 2 — Satellite Pointing Test Results\n",
        f"\n**Date:** {RUN_TS}\n",
        f"\n**Model:** `{MODEL_ID}` (HuggingFace Transformers, local)\n",
        f"\n**Input:** Satellite (NAIP RGB, 0.6m/pixel)\n",
        f"\n**DEM:** Cimarron River, 1m resolution, {dem_crs}\n",
        f"\n**Notebook:** `notebooks/attempt2/02_molmo2_satellite.ipynb`\n",
        f"\n**Attempt:** 2 (satellite imagery, HuggingFace Transformers local inference)\n",
        "\n---\n",
    ]

    for key, result in pointing_results.items():
        md_lines.append(f"\n## Pointing: {key.replace('_', ' ').title()}\n")
        md_lines.append(f"\n**Prompt:** {result['prompt']}\n")
        md_lines.append(f"\n**Points found:** {result['n_points']}\n")
        if result["pixel_coords"]:
            md_lines.append("\n**Pixel coordinates:**\n\n")
            for px, py, label in result["pixel_coords"]:
                md_lines.append(f"- ({px:.0f}, {py:.0f}) — {label}\n")
        if result["geo_coords"]:
            md_lines.append(f"\n**Geo-coordinates ({dem_crs}):**\n\n")
            for e, n, label in result["geo_coords"]:
                md_lines.append(f"- ({e:.1f}, {n:.1f}) — {label}\n")
        md_lines.append(f"\n**Raw output:**\n\n```\n{result['raw_output']}\n```\n")
        md_lines.append("\n---\n")

    # Summary table
    md_lines.append("\n## Summary\n\n")
    md_lines.append("| Test | Points | Notes |\n")
    md_lines.append("|------|--------|-------|\n")
    for key, result in pointing_results.items():
        n = result["n_points"]
        note = "coordinates parsed" if n > 0 else "no coordinates in output"
        md_lines.append(f"| {key} | {n} | {note} |\n")

    output_path = OUTPUT_DIR / f"{RUN_TS}_molmo2_satellite_results.md"
    with open(output_path, "w") as f:
        f.writelines(md_lines)

    print(f"\nResults saved to: {output_path}")